In [43]:
# import libraries
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import KFold, cross_val_score

In [44]:
# import feature data
# train_data = pd.read_csv('../data/processed_data/training_wildfire_weather_2020_2024.csv')
# test_data = pd.read_csv('../data/processed_data/test_wildfire_weather_2020_2024.csv')

In [45]:
# import feature data
# data = pd.read_csv('../data/processed_data/Wildfire_Weather_2020_2024.csv')

In [46]:
# Keep only the first occurrence of each unique fire_id
# data = data.drop_duplicates(subset='unique_id', keep='first')

In [47]:
# split data
# train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)

In [ ]:
# read in data
train_data = pd.read_csv('../data/processed_data/training_wildfire_weather_2020_2024.csv')
test_data = pd.read_csv('../data/processed_data/test_wildfire_weather_2020_2024.csv')


In [49]:
# Predictors and Targets
predictors = ['startdateseason', 'u10', 'v10', 'd2m', 't2m', 'msl', 'sp', 'lai_hv', 'lai_lv', 'tp', 'ssr']
target1 = 'size (acres)'
target2 = 'fire_spread (acres/day)'
target3 = 'duration'

In [50]:
# Split the data
X_train = train_data[predictors]
y1_train = train_data[target1]
y2_train = train_data[target2]
y3_train = train_data[target3]

X_test = test_data[predictors]
y1_test = test_data[target1]
y2_test = test_data[target2]
y3_test = test_data[target3]

In [51]:
# --- Step 4: One-hot encode the 'startdateseason' variable ---
X_train = pd.get_dummies(X_train, drop_first=True).astype(float)
X_test = pd.get_dummies(X_test, drop_first=True).astype(float)

# Ensure test set has same columns as train (fill missing with 0)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

## 5-fold CV baseline

In [52]:
# Set seed and CV strategy
random_seed = 42
kf = KFold(n_splits=5, shuffle=True, random_state=random_seed)

In [53]:
# Create a reusable function for baseline CV metrics
def compute_baseline_cv(X_train, y_train, target_name):
    dummy = DummyRegressor(strategy='mean')
    neg_mse = cross_val_score(dummy, X_train, y_train, cv=kf, scoring='neg_mean_squared_error')
    rmse = np.sqrt(-neg_mse)
    r2 = cross_val_score(dummy, X_train, y_train, cv=kf, scoring='r2')

    print(f"\n5-Fold CV Baseline Metrics for {target_name}:")
    print(f"Mean R²: {r2.mean():.4f}")
    print(f"Mean MSE: {-neg_mse.mean():.4f}")
    print(f"Mean RMSE: {rmse.mean():.4f}")

In [54]:
# Run for all three targets
compute_baseline_cv(X_train, y1_train, 'size (acres)')
compute_baseline_cv(X_train, y2_train, 'fire_spread (acres/day)')
compute_baseline_cv(X_train, y3_train, 'duration (days)')


5-Fold CV Baseline Metrics for size (acres):
Mean R²: -0.0002
Mean MSE: 155817425.2527
Mean RMSE: 12471.5097

5-Fold CV Baseline Metrics for fire_spread (acres/day):
Mean R²: -0.0005
Mean MSE: 1020724.2066
Mean RMSE: 994.6061

5-Fold CV Baseline Metrics for duration (days):
Mean R²: -0.0001
Mean MSE: 34.0602
Mean RMSE: 5.8360


## Test set baseline

In [55]:
# BASELINE PREDICTIONS: mean of training set
baseline1 = np.full_like(y1_test, fill_value=y1_train.mean(), dtype=np.float64)
baseline2 = np.full_like(y2_test, fill_value=y2_train.mean(), dtype=np.float64)
baseline3 = np.full_like(y3_test, fill_value=y3_train.mean(), dtype=np.float64)

In [56]:
def compute_baseline_metrics(y_true, y_pred, target_name):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    if target_name == 'duration':
        units = ' (days)'
    else:
        units = ''
    print(f"\nTest set baseline metrics for predicting {target_name + units}:")
    print(f"Mean training set target value: {y_pred[0]:.4f}")
    print(f"Baseline R²: {r2:.4f}")
    print(f"Baseline MSE: {mse:.4f}")
    print(f"Baseline RMSE: {rmse:.4f}")

# Evaluate baseline performance
compute_baseline_metrics(y1_test, baseline1, target1)
compute_baseline_metrics(y2_test, baseline2, target2)
compute_baseline_metrics(y3_test, baseline3, target3)


Test set baseline metrics for predicting size (acres):
Mean training set target value: 4878.1695
Baseline R²: -0.0002
Baseline MSE: 130787160.1986
Baseline RMSE: 11436.2214

Test set baseline metrics for predicting fire_spread (acres/day):
Mean training set target value: 403.0489
Baseline R²: -0.0002
Baseline MSE: 828809.0401
Baseline RMSE: 910.3895

Test set baseline metrics for predicting duration (days):
Mean training set target value: 12.9265
Baseline R²: -0.0002
Baseline MSE: 34.5732
Baseline RMSE: 5.8799
